In [ ]:
# Aut-min unsolved124 × s20_mk2 @ 1M / mrl 64 — CONFIG
# Edit ONLY CHUNK_INDEX if needed (pre-set per file).
#
# Starts = Aut-min class reps (unsolved_124_aca_classes.csv), unchanged.
# Arm = s20_mk2 = L+20S+2MK only. Primary metric: min_relator_length vs start_total.
# HIGH_SPEEDUP = N_WORKERS="auto" + ENGINE="hcompact" (no separate boolean).
# On ~51 GB / 8-core Colab, memory-caps to ~6 workers at 1M — never pin 8.

REPO_URL   = "https://github.com/Avi161/ACSolverX.git"
REPO_DIR   = "ACSolverX"
BRANCH     = "cursor/heur-u124-s20mk2-a42e"
CLONE      = True
UPDATE_REPO = True

MOUNT_DRIVE = True
DRIVE_DIR   = "/content/drive/MyDrive/acsolverx/hsearch_u124_s20mk2_1m"

CHUNK_INDEX = 1

cfg = dict(
    DATASET   = "unsolved124",
    SUBSET    = None,

    ARMS      = ["s20_mk2"],

    CHUNKS       = 4,
    CHUNK_INDEX  = CHUNK_INDEX,

    ENGINE       = "hcompact",
    N_WORKERS    = "auto",
    KEEP_PATH    = False,

    NODE_BUDGET = 1_000_000,
    CHECKPOINTS = [1000, 5000, 10000, 25000, 50000, 100000, 250000, 500000, 1000000],
    MAX_RELATOR_LENGTH = 64,
    RESUME    = True,
    OUT_STEM  = "hsearch_u124_s20mk2",
    STAGE_DIR = "/content/hsearch_stage",
)

HEARTBEAT_SECS = 60
PROGRESS_SECS  = 300


In [ ]:
# ==================== SETUP (clone / pull / mount / purge) ================
import os, sys, subprocess, importlib

def sh(cmd):
    print("$", cmd)
    p = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if p.stdout: print(p.stdout[-2000:])
    if p.returncode != 0 and p.stderr: print("STDERR:", p.stderr[-2000:])

try:
    import google.colab  # noqa
    IN_COLAB = True
except Exception:
    IN_COLAB = False
print("Colab:", IN_COLAB)

if IN_COLAB:
    BASE = "/content"
    os.chdir(BASE)
    if not os.path.isdir(REPO_DIR):
        if CLONE:
            sh(f"git clone --branch {BRANCH} {REPO_URL} {REPO_DIR}")
    elif UPDATE_REPO:
        sh(f"cd {REPO_DIR} && git fetch origin {BRANCH} && git reset --hard FETCH_HEAD")
    sh(f"cd {REPO_DIR} && git log -1 --oneline")
    sh("pip -q install numba numpy")
    if MOUNT_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        os.makedirs(DRIVE_DIR, exist_ok=True)
    os.makedirs(cfg.get("STAGE_DIR", "/content/hsearch_stage"), exist_ok=True)
    REPO_ROOT = os.path.join(BASE, REPO_DIR)
else:
    REPO_ROOT = os.getcwd()
    while REPO_ROOT != "/" and not (
        os.path.isdir(os.path.join(REPO_ROOT, "experiments"))
        and os.path.isdir(os.path.join(REPO_ROOT, "data"))
    ):
        REPO_ROOT = os.path.dirname(REPO_ROOT)

os.chdir(REPO_ROOT)
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("repo root:", REPO_ROOT)
print("BRANCH:", BRANCH, "ENGINE:", cfg["ENGINE"], "N_WORKERS:", cfg["N_WORKERS"])
assert cfg["ENGINE"] == "hcompact", "ENGINE=hcompact required for HIGH_SPEEDUP @1M"
assert BRANCH, "BRANCH must be set"
import subprocess as _sp
_here = _sp.check_output(
    ["git", "rev-parse", "--abbrev-ref", "HEAD"], text=True).strip()
print("git HEAD branch:", _here, "(Colab clones BRANCH; local may differ)")

for _m in [m for m in sys.modules if m == "experiments" or m.startswith("experiments.")]:
    del sys.modules[_m]
importlib.invalidate_caches()

from experiments.heuristic_search.core.hsolve import greedy_search_h
from experiments.heuristic_search.runners import run_unsolved124_s20mk2 as _ru
_ = greedy_search_h("xyx", "yx", 20, max_relator_length=32,
                    config=_ru.run_ab.ARMS["s20_mk2"])
print("kernels warm — setup done")
print("tip: Runtime → Restart → Run All resumes (UPDATE_REPO + flock + RESUME)")
print("ARMS:", cfg["ARMS"], "chunk:", cfg["CHUNK_INDEX"], "/", cfg["CHUNKS"])


In [ ]:
# ==================== RUN (HIGH_SPEEDUP multi-worker) =====================
from experiments.heuristic_search.runners.run_unsolved124_s20mk2 import (
    run_unsolved124_s20mk2)
from experiments.heuristic_search.runners import run_ab as _ra
nw, per = _ra._resolve_workers(
    {"N_WORKERS": cfg["N_WORKERS"]}, cfg["NODE_BUDGET"],
    cfg["MAX_RELATOR_LENGTH"], cfg["ENGINE"], cfg["KEEP_PATH"])
print(f"HIGH_SPEEDUP resolve: N_WORKERS={cfg['N_WORKERS']} -> {nw} workers "
      f"(~{per:.1f} GB/search est., ENGINE={cfg['ENGINE']})")
print(f"budget={cfg['NODE_BUDGET']:,} mrl={cfg['MAX_RELATOR_LENGTH']} "
      f"chunk={cfg['CHUNK_INDEX']}/{cfg['CHUNKS']} arms={cfg['ARMS']}")
run_unsolved124_s20mk2(
    cfg,
    out_dir=(DRIVE_DIR if (IN_COLAB and MOUNT_DRIVE) else
             "results/heuristic_search/u124_s20mk2_1m"),
    heartbeat_secs=HEARTBEAT_SECS,
    progress_secs=PROGRESS_SECS,
)
print("done — leave session up until jsonl mirror finishes.")
print("After all 4 chunks: merge_colab_chunks on the Drive dir (see README).")
